# 🧬 PANACEE – Notebook 03 : Phase 3 – Multi-propriétés & Analyse
## Prédiction simultanée de 17 propriétés + analyse combinatoire

**Objectif :** Entraîner un modèle qui prédit simultanément :

| Groupe | Propriété | Type |
|--------|-----------|------|
| Toxicité | 12 cibles Tox21 | Classification |
| Efficacité | Score bioactivité | Classification |
| Solubilité | logS | Régression |
| Lipophilicité | logD | Régression |
| Biodisponibilité | Oral BA | Classification |
| Stabilité métabolique | CLint | Classification |

**Algorithmes avancés inclus :**
- 🔍 MCTS (Monte Carlo Tree Search) pour combinaisons optimales
- 🎯 Optimisation bayésienne des doses
- 📐 Front de Pareto multi-objectif
- 🤝 Analyse de synergie via MolecularReasoner

---
> **Durée estimée** : 2-3h sur GPU T4

In [ ]:
import sys, os
sys.path.insert(0, "/content/panacee")
os.chdir("/content/panacee")

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from config import DEVICE, PHASE3, CHECKPOINT_DIR, GNN_ARCHITECTURE

print(f"Device : {DEVICE}")
print(f"Arch   : {GNN_ARCHITECTURE.upper()}")

## Étape 1 – Récupérer le checkpoint Phase 2

In [ ]:
PHASE2_CKPT = f"{CHECKPOINT_DIR}/phase2/panacee_phase2.pth"

if not os.path.exists(PHASE2_CKPT):
    print("Récupération depuis Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    import shutil
    src = "/content/drive/MyDrive/Panacee_Checkpoints/panacee_phase2.pth"
    if os.path.exists(src):
        os.makedirs(os.path.dirname(PHASE2_CKPT), exist_ok=True)
        shutil.copy(src, PHASE2_CKPT)
        print(f"✅ Checkpoint Phase 2 récupéré")
    else:
        print("❌ Checkpoint Phase 2 non trouvé. Lancez d'abord Notebook_02.")
else:
    print(f"✅ Checkpoint Phase 2 disponible : {PHASE2_CKPT}")

## Étape 2 – Téléchargement des datasets multi-propriétés

In [ ]:
from data_loaders import download_all_phase3_data, merge_phase3_datasets
from config import DATA_DIR

print("Téléchargement des 6 datasets Phase 3...")
print("(ESOL, Lipophilicity, BBBP, ClinTox, HIV + Tox21 déjà téléchargé)")
print()

dataset_paths = download_all_phase3_data(save_dir=DATA_DIR)

print(f"\n✅ Datasets téléchargés :")
for name, path in dataset_paths.items():
    if path and os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  {name:<20}: {len(df):,} molécules | {df.shape[1]} colonnes")
    else:
        print(f"  {name:<20}: ⚠️  Non disponible")

In [ ]:
# Fusionner tous les datasets en un seul DataFrame multi-propriétés
print("Fusion des datasets...")
merged_df, merged_path = merge_phase3_datasets(dataset_paths, save_dir=DATA_DIR)

print(f"\n✅ Dataset fusionné : {merged_df.shape[0]:,} molécules × {merged_df.shape[1]} colonnes")
print(f"   Sauvegardé : {merged_path}")
print(f"\nTaux de complétion par colonne :")
for col in merged_df.columns:
    pct = merged_df[col].notna().mean() * 100
    bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
    print(f"  {col:<25}: {pct:5.1f}% |{bar}|")

## Étape 3 – DataLoaders multi-propriétés

In [ ]:
from data_loaders import make_multi_property_loaders
from config import PHASE3

train_loader, val_loader = make_multi_property_loaders(
    df=merged_df,
    batch_size=PHASE3["batch_size"],
    val_split=0.10,
)

print("DataLoaders Phase 3 :")
print(f"  Entraînement : {len(train_loader.dataset):,} molécules")
print(f"  Validation   : {len(val_loader.dataset):,} molécules")
print(f"  Taille batch : {PHASE3['batch_size']}")

# Exemple de batch
sample_batch, sample_labels = next(iter(train_loader))
print(f"\nExemple de batch :")
print(f"  Graphes : {sample_batch.num_graphs}")
for prop, tensor in sample_labels.items():
    valid_pct = (~tensor.isnan()).float().mean().item() * 100
    print(f"  {prop:<25}: shape={tensor.shape}, valides={valid_pct:.1f}%")

## Étape 4 – Entraînement Phase 3

In [ ]:
from trainer import train_phase3

print("=" * 65)
print("LANCEMENT PHASE 3 – MULTI-PROPRIÉTÉS")
print("=" * 65)
print(f"Epochs      : {PHASE3['epochs']}")
print(f"LR encodeur : {PHASE3['lr_encoder']}")
print(f"LR têtes    : {PHASE3['lr_heads']}")
print()

phase3_ckpt = train_phase3(
    train_loader=train_loader,
    val_loader=val_loader,
    pretrained_tox_path=PHASE2_CKPT,
    arch=GNN_ARCHITECTURE,
)

print(f"\n✅ Phase 3 terminée !")
print(f"   Checkpoint : {phase3_ckpt}")

## Étape 5 – Prédictions et analyse

In [ ]:
from predictor import PanaceePredictor

predictor = PanaceePredictor.load(phase3_ckpt, arch=GNN_ARCHITECTURE)

# Molécules test bien connues
test_molecules = {
    "Paracétamol"   : "CC(=O)Nc1ccc(O)cc1",
    "Aspirine"      : "CC(=O)Oc1ccccc1C(=O)O",
    "Ibuprofène"    : "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "Caféine"       : "Cn1c(=O)c2c(ncn2C)n(c1=O)C",
    "Pénicilline G" : "CC1(SC2C(NC1=O)C(=O)N2Cc1ccccc1)C(=O)O",
}

print("=" * 70)
print("PRÉDICTIONS PANACEE – MOLÉCULES TEST")
print("=" * 70)

results = {}
for name, smi in test_molecules.items():
    r = predictor.predict(smi)
    results[name] = r
    if "error" not in r:
        print(f"\n{name} ({smi[:30]}...)")
        print(f"  Score global     : {r['global_score']:.4f}")
        print(f"  Toxicité max     : {r['toxicity_max']:.4f}")
        print(f"  Efficacité       : {r['efficacy']:.4f}")
        print(f"  Solubilité (logS): {r['solubility']:.4f}")
        print(f"  Biodisponibilité : {r['bioavailability']:.4f}")

In [ ]:
# Visualisation en radar chart
import matplotlib.patches as mpatches

categories = ["Efficacité", "Sécurité\n(1-Tox)", "Solubilité", "Biodispo.", "Stab.\nMéta"]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

colors = ["royalblue", "tomato", "forestgreen", "darkorange", "purple"]

for (name, r), color in zip(results.items(), colors):
    if "error" in r:
        continue
    sol_norm = max(0, min(1, (r["solubility"] + 6) / 6))  # normaliser logS [-6,0] → [0,1]
    values = [
        r["efficacy"],
        1 - r["toxicity_max"],
        sol_norm,
        r["bioavailability"],
        r["metabolic_stability"],
    ] + [r["efficacy"]]  # boucle fermée
    
    ax.plot(angles, values, color=color, linewidth=2, label=name)
    ax.fill(angles, values, color=color, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=11)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], size=8)
ax.set_ylim(0, 1)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=10)
ax.set_title("Profils moléculaires – Radar Chart", size=14, pad=20)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/panacee/results/radar_chart.png", dpi=150, bbox_inches="tight")
plt.show()

## Étape 6 – Analyse combinatoire avancée

In [ ]:
from advanced_algorithms import MCTSCombinationSearch, ParetoOptimizer

smiles_list = list(test_molecules.values())

# Fonction de score via le predictor
def combo_score_fn(indices):
    smi_combo = [smiles_list[i] for i in indices]
    result    = predictor.analyze_combination(smi_combo)
    return result.get("combo_score", 0.0)

print("Recherche MCTS des meilleures combinaisons...")
mcts = MCTSCombinationSearch(
    score_fn=combo_score_fn,
    n_simulations=100,
    max_combo_size=3,
)
best_indices, best_score = mcts.search(list(range(len(smiles_list))))

names = list(test_molecules.keys())
print(f"\n✅ Meilleure combinaison MCTS :")
for i in best_indices:
    print(f"  - {names[i]}")
print(f"  Score combinaison : {best_score:.4f}")

In [ ]:
# Visualisation du front de Pareto Efficacité vs Sécurité
efficacy_scores = [r.get("efficacy", 0)             for r in results.values() if "error" not in r]
safety_scores   = [1 - r.get("toxicity_max", 1)     for r in results.values() if "error" not in r]
names_valid     = [n                                 for n, r in results.items() if "error" not in r]

objectives = np.column_stack([efficacy_scores, safety_scores])
pareto_idx = ParetoOptimizer.pareto_front(objectives)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(efficacy_scores, safety_scores, s=120, zorder=5, color="steelblue", label="Toutes molécules")

# Annoter les points
for i, name in enumerate(names_valid):
    ax.annotate(name, (efficacy_scores[i], safety_scores[i]),
                xytext=(8, 4), textcoords="offset points", fontsize=9)

# Mettre en évidence le front de Pareto
pf_x = [efficacy_scores[i] for i in pareto_idx]
pf_y = [safety_scores[i]   for i in pareto_idx]
ax.scatter(pf_x, pf_y, s=200, zorder=6, color="tomato",
           edgecolors="darkred", linewidths=2, label="Front de Pareto")

ax.set_xlabel("Score Efficacité", fontsize=12)
ax.set_ylabel("Score Sécurité (1 - Toxicité max)", fontsize=12)
ax.set_title("Front de Pareto : Efficacité vs Sécurité", fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/panacee/results/pareto_front.png", dpi=150)
plt.show()

print("\nMolécules sur le front de Pareto (meilleur compromis efficacité/sécurité) :")
for i in pareto_idx:
    print(f"  - {names_valid[i]} (eff={efficacy_scores[i]:.3f}, sec={safety_scores[i]:.3f})")

## Étape 7 – Génération du rapport final

In [ ]:
report_path = "/content/panacee/results/rapport_final.txt"
report = predictor.generate_report(
    smiles_list=list(test_molecules.values()),
    output_path=report_path,
)

print(report)
print(f"\n✅ Rapport sauvegardé : {report_path}")

In [ ]:
# Sauvegarde finale sur Google Drive
import shutil
drive_dir = "/content/drive/MyDrive/Panacee_Checkpoints"

# Checkpoint Phase 3
shutil.copy(phase3_ckpt, os.path.join(drive_dir, "panacee_phase3.pth"))
print("✅ Checkpoint Phase 3 sauvegardé")

# Résultats
results_drive = os.path.join(drive_dir, "results")
os.makedirs(results_drive, exist_ok=True)
for f in os.listdir("/content/panacee/results"):
    shutil.copy(
        os.path.join("/content/panacee/results", f),
        os.path.join(results_drive, f)
    )
print("✅ Résultats sauvegardés sur Drive")
print()
print("=" * 60)
print("🎉 ENTRAÎNEMENT PANACEE COMPLET !")
print("=" * 60)
print("   Phase 1 : Pré-entraînement MGM ✅")
print("   Phase 2 : Fine-tuning Toxicité ✅")
print("   Phase 3 : Multi-propriétés     ✅")
print()
print("   Pour prédire de nouvelles molécules :")
print("   >>> predictor = PanaceePredictor.load('panacee_phase3.pth')")
print("   >>> result = predictor.predict('votre_smiles_ici')")